# 1.2 Data for the Survival Analysis

**Core series, class 2.** In **3.4** we'll ask: **after a type 2 diabetes diagnosis, how long until people develop chronic kidney disease (CKD)?** Kidney damage is one of the most common complications of diabetes. Answering a "how long until" question needs more than "who has it": it needs **dates**, and a careful idea of how long we could **see** each person in their health records. This class gathers those dates.

### By the end of this notebook you'll be able to

1. Explain **time zero**, **event**, **censoring**, and **follow-up**, the building blocks of a survival analysis
2. Explain why follow-up should come from **health record dates**
3. Find the first date of a diagnosis with SQL (`MIN()` and `GROUP BY`)
4. Check a concept ID before using it
5. Describe how many people had the event, and how long they were followed

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Among people with type 2 diabetes, who developed CKD **after** their diagnosis, and how long was everyone followed? |
| 🧪 **The analysis** | A SQL query for each person's **first CKD date**; then, for each case, time from the first diabetes date to CKD, or to the end of their records |
| 🎯 **Why** | A survival analysis compares **time to an event**, so it needs a clear start, a clear event, and a fair end for people without the event (**censoring**) |
| ✅ **What we conclude** | How many cases had CKD after diabetes, how many before (they can't be used), and how long people were followed |
| ⚠️ **What it can't tell us** | Risk or what predicts it: that's **3.4** |
| 🔗 **Connections** | Uses the sample from **1.1**; joined to the other data in **2.3**; analysed in **3.4** |

| | |
|---|---|
| 📂 **Uses** | `cohort_sample.csv` from **1.1**, and the Controlled Tier dataset (BigQuery) |
| 💾 **Saves** | `ckd_dates.csv` |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts: survival data | 20 |
| 2. Setup | 5 |
| 3. Check the concept | 5 |
| 4. First CKD dates | 10 |
| 5. Time zero, event, censoring | 15 |
| Wrap-up | 5 |

**Before you start:** run **1.1**, which saves `cohort_sample.csv`.

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 60 minutes |
| ⏱️ **Run time** | about **3 minutes** to run every cell. Slowest: the CKD query (1–2 min) |
| 💰 **Data cost** | **under $1**: two small queries |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| 💡 **To spend less** | the CKD dates are saved, so later classes don't query again |
| **Estimated total** | about **$1** per student for this class |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts: survival data

A survival analysis follows people **over time** and asks **how long** until an **event** happens. The event doesn't have to be death: here it's a CKD diagnosis.

| Term | Means | In our study |
|---|---|---|
| **Time zero** | when each person's clock starts | the **first type 2 diabetes** date |
| **Event** | what we're waiting for | the **first CKD** date, after time zero |
| **Censoring** | the clock stops **without** the event: we stop being able to see the person | the earlier of their **last EHR visit** and **death** |
| **Follow-up time** | time zero to the event, or to censoring | in years |

### Why censoring matters

Someone whose records end 2 years after their diabetes diagnosis, without CKD, **didn't** go 2 years and then stay CKD-free forever: we just **stopped seeing** them. Censoring lets the analysis use the 2 years we **did** see, without pretending to know what happened after. Ignoring censoring (e.g. counting everyone without CKD as "never got it") underestimates risk.

```
diabetes      last EHR visit
   │─────────────────│           censored: no CKD while we could see them
   │──────────X                  event: CKD here
CKD ──X  diabetes                excluded: CKD came first
```

### Why end follow-up at the last **EHR** visit?

A diagnosis can only appear in the records while someone is getting care at a site that shares data with *All of Us*. After their **last EHR visit**, a CKD diagnosis elsewhere would be invisible. We use `last_ehr_date` from the cohort query, not *All of Us*'s `observation_period` table, because that one also counts **survey** dates, which can stretch follow-up past the last health record.

### Who can't be in the analysis

People whose CKD was recorded **on or before** their first diabetes date. CKD can't be "after diabetes" for them, and it may have other causes. We count them, then exclude them in 3.4.

### Glossary

| Term | Means |
|---|---|
| **Survival analysis** | methods for time-to-event data, with censoring |
| **Kaplan–Meier curve** | the share of people still event-free over time (3.4) |
| **Hazard** | the event rate at a given moment, among those still at risk (3.4) |
| **CKD** | chronic kidney disease: kidney damage or reduced function for 3 months or more |

## Part 2. Setup

Load the settings, then the sample saved in **1.1**. `col_types = cols(person_id = "c")` keeps the IDs as text. Dates in the file (like `2019-04-12`) are read as **dates** automatically.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

cohort_sample <- load_data("cohort_sample.csv", col_types = cols(person_id = "c")) %>%
  mutate(across(c(first_t2d_date, death_date, first_ehr_date, last_ehr_date), as.Date))   # dates, even if a column is all empty
cohort_sample %>% count(cohort)

ids_sql <- str_c(cohort_sample$person_id, collapse = ", ")   # don't print: it contains IDs

## Part 3. Check the concept

Before using a concept ID, **look it up**: a typo in one digit gives a completely different disease, and the query would still run. The `concept` table has each ID's name, vocabulary, and whether it's a **standard** concept (`S`), the kind used in `condition_occurrence`.

We also count how many concepts are **under** CKD in the hierarchy (`concept_ancestor`).

In [ ]:
CKD_CONCEPT <- 46271022   # Chronic kidney disease

run_query(str_glue("
SELECT concept_id, concept_name, vocabulary_id, standard_concept
FROM `{CURRENT_DATASET}.concept`
WHERE concept_id IN ({CKD_CONCEPT}, 201826)
"))

run_query(str_glue("
SELECT COUNT(*) AS concepts_under_ckd
FROM `{CURRENT_DATASET}.concept_ancestor`
WHERE ancestor_concept_id = {CKD_CONCEPT}
"))

**🔍 Read the output:** `46271022` should be **Chronic kidney disease** (SNOMED, standard `S`), and `201826` **Type 2 diabetes mellitus**. The second number is how many concepts count as CKD through the hierarchy: the stages (stage 1 to 5), CKD due to diabetes or high blood pressure, and so on. If the name isn't what you expect, **stop** and check the ID at [athena.ohdsi.org](https://athena.ohdsi.org/).

## Part 4. First CKD dates

For each person in our sample with any CKD diagnosis: the **first** date (`MIN()`), and the number of different days with a CKD code (`COUNT(DISTINCT …)`). `GROUP BY co.person_id` gives one row per person, like `group_by()` + `summarise()` in R.

💰 Cents to under $1; a minute or two.

In [ ]:
ckd_sql <- str_glue("
SELECT co.person_id,
       MIN(co.condition_start_date)          AS first_ckd_date,
       COUNT(DISTINCT co.condition_start_date) AS ckd_code_days
FROM `{CURRENT_DATASET}.condition_occurrence` AS co
JOIN `{CURRENT_DATASET}.concept_ancestor` AS ca
  ON co.condition_concept_id = ca.descendant_concept_id
WHERE ca.ancestor_concept_id = {CKD_CONCEPT}
  AND co.person_id IN ({ids_sql})
GROUP BY co.person_id
")

ckd_dates <- run_query(ckd_sql) %>% mutate(person_id = as.character(person_id))

nrow(ckd_dates)   # people in our sample with any CKD diagnosis

# CKD by group
cohort_sample %>%
  mutate(has_ckd = person_id %in% ckd_dates$person_id) %>%
  count(cohort, has_ckd) %>%
  hide_small_counts()

save_data(ckd_dates, "ckd_dates.csv")

**🔍 Read the output:** CKD is **much more common among cases** than controls, as expected: diabetes is the leading cause of kidney disease. In our test run: 28% of cases (1,377 of 5,000) vs 7% of controls (328 of 5,000). This table counts CKD at **any** time, before or after diabetes; Part 5 sorts that out.

## Part 5. Time zero, event, censoring

We put the pieces together for the **cases**, as a preview of 3.4. `left_join()` adds each case's first CKD date (or `NA` if they never had one; joins are covered in **2.3**).

| New column | Calculated as |
|---|---|
| `end_of_records` | the earlier of `last_ehr_date` and `death_date`: `pmin(…, na.rm = TRUE)` takes the smaller value, ignoring missing ones |
| `timing` | *CKD before or on diabetes date*, *CKD after diabetes*, or *no CKD* (`case_when()`: the first matching rule wins) |
| `followup_years` | from the first diabetes date to the CKD date (if after), or to the end of records |

In [ ]:
cases <- cohort_sample %>%
  filter(cohort == "case") %>%
  left_join(ckd_dates, by = "person_id") %>%
  mutate(
    end_of_records = pmin(last_ehr_date, death_date, na.rm = TRUE),
    timing = case_when(
      !is.na(first_ckd_date) & first_ckd_date <= first_t2d_date ~ "CKD before or on diabetes date",
      !is.na(first_ckd_date)                                     ~ "CKD after diabetes",
      TRUE                                                       ~ "no CKD"
    ),
    followup_end   = if_else(timing == "CKD after diabetes", first_ckd_date, end_of_records),
    followup_years = as.numeric(followup_end - first_t2d_date) / 365.25
  )

cases %>% count(timing) %>% hide_small_counts()

cases %>%
  filter(timing != "CKD before or on diabetes date") %>%
  group_by(timing) %>%
  summarise(people = n(), median_years = round(median(followup_years), 1),
            pct_under_1_year = round(100 * mean(followup_years < 1), 1))

**🔍 Read the output:**

- **CKD before or on the diabetes date:** these people can't be used (their "event" came before time zero). Often CKD and diabetes are first recorded **on the same day**, e.g. when someone's records start with a visit that lists both.
- **CKD after diabetes:** the **events**. `median_years` is the typical time from diabetes to CKD.
- **No CKD:** **censored**. Their `median_years` is how long we could see them, CKD-free.
- `pct_under_1_year`: people followed less than a year contribute little information.

In our test run, among 5,000 cases: **925** had CKD after diabetes (median 4.4 years later), **452** had CKD before or on the diabetes date, and **3,623** had no CKD (followed for a median of 5.2 years). About 1 in 6 to 1 in 5 were followed for less than a year.

**💬 Discuss:** why might someone's first diabetes date in the records be **later** than when they actually developed diabetes? What does that do to time zero? (Records often start when someone joins a health system, so the "first" diagnosis in the data may be years after the real one. This is called **left truncation**, and it's one reason EHR-based survival analyses need care.)

### ✍️ Your turn

How many cases had **CKD and diabetes recorded on the same day**? (A subset of "before or on".)

<details>
<summary><b>Show answer</b></summary>

```r
cases %>%
  filter(!is.na(first_ckd_date)) %>%
  count(same_day = first_ckd_date == first_t2d_date) %>%
  hide_small_counts()
```

`TRUE` is the same-day count. If it's a large share of "before or on", that supports the idea that many records simply **start** with both diagnoses already present.

</details>

In [ ]:
# Your code here

## ⚠️ Common errors

| Message | Means | Fix |
|---|---|---|
| *'cohort_sample.csv' does not exist* | 1.1 hasn't been run, or `BUCKET_NAME` is different | Run 1.1, or check `aou_setup.R` |
| *Query exceeds … maximum length* | Too many IDs in `IN (…)` | Use the 1.1 sample (10,000 people) |
| Dates are text, and subtracting them fails | The CSV's date columns weren't recognised as dates | `mutate(across(ends_with("_date"), as.Date))` |
| `followup_years` is negative | A last EHR date before the first diabetes date (rare data errors) | Check those people; 3.4 drops them |

## ✅ Key takeaways

- A survival analysis needs **time zero**, an **event**, and **censoring**
- End follow-up at the **last health record** (or death), not at today's date
- **Check every concept ID** by name before using it
- People whose event came **before** time zero can't be used
- Records starting **late** can make time zero wrong (left truncation)

**Next:** **2.1 clean**: tidy the survey answers and measurements from 1.1.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).